In [1]:
import pandas as pd
# Excelファイルを読み込む
df = pd.read_excel(r"C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\Merge_20250815_1.xlsx")
print(df.columns)

Index(['INDEX', '在院日数', '転帰', '平均気温', '平均気圧', 'Month', 'Age', 'Male', 'BMI',
       'NIHSS_aa意識レベル', 'NIHSS_ab質問に対する反応', 'NIHSS_ac命令への反応', 'NIHSS_b最良の注視',
       'NIHSS_c視野', 'NIHSS_d顔面麻痺', 'NIHSS_e上肢の運動右', 'NIHSS_f上肢の運動左',
       'NIHSS_g下肢の運動右', 'NIHSS_h下肢の運動左', 'NIHSS_i四肢の運動失調', 'NIHSS_j感覚',
       'NIHSS_k言語', 'NIHSS_l構音障害', 'NIHSS_m消去無視', 'NIHSS_total score',
       'NIHSS_total_初診時', 'NIHSS_total_24h後', 't-pa', 'エダラボン', '抗てんかん剤', 'RAS',
       'Asprin', 'P2Y12', '抗凝固薬', '糖尿病治療薬', 'スタチン', 'β遮断薬', '抗生剤', '精神薬',
       '脳血栓回収術', 'ad_APTT', 'ad_Alb', 'ad_BNP', 'ad_BUN', 'ad_CRP', 'ad_Hb',
       'ad_HbA1c', 'ad_K', 'ad_LDL_C', 'ad_Na', 'ad_PT_INR', 'ad_WBC',
       'ad_eGFR', 'hours_48_APTT', 'hours_48_Alb', 'hours_48_BNP',
       'hours_48_BUN', 'hours_48_CRP', 'hours_48_Hb', 'hours_48_HbA1c',
       'hours_48_K', 'hours_48_LDL_C', 'hours_48_Na', 'hours_48_PT_INR',
       'hours_48_WBC', 'hours_48_eGFR', 'hours_48_心拍数', 'hours_48_非観血_収縮期',
       'TIA', 'アテローム血栓性梗塞', 'その他の脳梗塞', 'ラクナ

In [5]:
# -*- coding: utf-8 -*-
# ============================================================
# train_block_optimal.py  （修正版：SHAP/feature名ズレ解消 + QUICKモード）
# ============================================================

from __future__ import annotations

import os, sys, warnings, logging, inspect, platform, subprocess
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
import xgboost as xgb
from xgboost.core import XGBoostError

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    GroupKFold, StratifiedKFold, KFold, train_test_split
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    r2_score, median_absolute_error, mean_squared_error,
    average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor
import joblib

# ───────────────────────────────
# 基本設定
# ───────────────────────────────
RUN_STAGE = "FINAL"  # ← 挙動確認用。負荷を下げて成果物まで出力。("TUNE", "FINAL", "ALL" も可)
RAW_PATH = r"C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\Merge_20250815_1.xlsx"

class Config:
    RESULT_DIR = Path(f"results_{datetime.now():%Y%m%d_%H%M}")
    TARGET_COL = "転帰"
    LOS_COL = "在院日数"
    INDEX_COL = "INDEX"
    TIME_FLAG_COL = "時間検証"
    CLASS_NAMES = ["自宅", "その他"]
    RANDOM_STATE = 42
    CAL_METHOD = "sigmoid"
    CAL_N_BINS = 10
    LOS_BIN_COL = "LOS_bin"
    LOS_CUTOFF_DAYS = 21
    LOS_WINSOR_MAX = 60
    PRED_DIRNAME = "predictions"
    ARTIFACT_DIRNAME = "artifacts"

# ★ TUNE / FINAL で切り替えるパラメタ
class TierTune:
    MI_M = 3
    MI_MAX_ITER = 15
    N_SPLITS = 4
    N_TRIALS = 30
    PATIENCE = 10
    N_BOOT = 300
    ES_ROUNDS = 40
    XGB_N_EST_CLS = 500
    XGB_N_EST_REG = 700
    XGB_MAX_BIN = 128

class TierFinal:
    MI_M = 15
    MI_MAX_ITER = 25
    N_SPLITS = 5
    N_TRIALS = 80
    PATIENCE = 25
    N_BOOT = 1200
    ES_ROUNDS = 120
    XGB_N_EST_CLS = 1800
    XGB_N_EST_REG = 1200
    XGB_MAX_BIN = 128

# ★ 追加：QUICK（さらに軽量、挙動確認用）
class TierQuick:
    MI_M = 1
    MI_MAX_ITER = 5
    N_SPLITS = 3
    N_TRIALS = 5
    PATIENCE = 3
    N_BOOT = 50
    ES_ROUNDS = 10
    XGB_N_EST_CLS = 200
    XGB_N_EST_REG = 300
    XGB_MAX_BIN = 64

GLOBAL_FORCE_KEEP_COLS = ["NIHSS_total_初診時"]

# ─────────────────────────────────────────────
# ログと環境
# ─────────────────────────────────────────────
Config.RESULT_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(Config.RESULT_DIR / "training.log", encoding="utf-8"),
              logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

def save_environment_snapshot(out_dir: Path):
    try:
        txt = []
        txt.append(f"Python: {sys.version}")
        txt.append(f"Platform: {platform.platform()}")
        txt.append(f"xgboost: {xgb.__version__}")
        try:
            import sklearn
            txt.append(f"scikit-learn: {sklearn.__version__}")
        except Exception:
            pass
        try:
            pkgs = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
            (out_dir / "pip_freeze.txt").write_text(pkgs, encoding="utf-8")
        except Exception as e:
            txt.append(f"pip freeze failed: {e}")
        (out_dir / "env_info.txt").write_text("\n".join(txt), encoding="utf-8")
    except Exception as e:
        logger.warning(f"save_environment_snapshot failed: {e}")

save_environment_snapshot(Config.RESULT_DIR)
np.random.seed(Config.RANDOM_STATE)

# ─────────────────────────────────────────────
# GPU / 速度最適化
# ─────────────────────────────────────────────
def xgb_gpu_params() -> dict:
    ver = tuple(int(x) for x in xgb.__version__.split(".")[:2])
    if ver >= (2, 0):
        return dict(tree_method="hist", device="cuda", predictor="gpu_predictor")
    else:
        try:
            import cupy  # noqa
            return dict(tree_method="gpu_hist", predictor="gpu_predictor")
        except Exception:
            return dict(tree_method="hist", predictor="auto")

def fit_with_gpu_fallback_xgb(model, X, y, sample_weight=None, fit_kwargs=None):
    kw = dict(fit_kwargs or {})
    if "sample_weight" in kw:
        kw.pop("sample_weight")
    if sample_weight is not None:
        kw["sample_weight"] = sample_weight
    try:
        model.fit(X, y, **kw)
        return model, False
    except XGBoostError:
        try:
            model.set_params(**{k: v for k, v in dict(model.get_params()).items()
                                if k not in ("device", "tree_method", "predictor")})
        except Exception:
            pass
        model.set_params(tree_method="hist", predictor="auto")
        model.fit(X, y, **kw)
        return model, True

def _build_es_kwargs_model(model, X_val, y_val, es_rounds: int):
    sig = inspect.signature(model.fit)
    kw = {}
    if "eval_set" in sig.parameters:
        kw["eval_set"] = [(X_val, y_val)]
    if "early_stopping_rounds" in sig.parameters:
        kw["early_stopping_rounds"] = es_rounds
    if isinstance(model, (xgb.XGBClassifier, xgb.XGBRegressor)):
        kw["verbose"] = False
    return kw

# ─────────────────────────────────────────────
# 前処理
# ─────────────────────────────────────────────
class VIFSelector(BaseEstimator, TransformerMixin):
    def __init__(self, thr: float = 8.0, max_iter: int = 30, add_constant: bool = True, protected: List[str] | None = None):
        self.thr = float(thr); self.max_iter = int(max_iter); self.add_constant = bool(add_constant)
        self.protected = set(protected or []); self.keep_columns_: List[str] = []

    def _compute_vif(self, Xdf: pd.DataFrame) -> pd.DataFrame:
        X_ = Xdf.copy()
        if self.add_constant:
            X_ = X_.assign(__const__=1.0)
        vifs = []
        for i, c in enumerate(Xdf.columns):
            try:
                v = variance_inflation_factor(X_.values, i)
            except Exception:
                v = np.nan
            vifs.append((c, float(v)))
        return pd.DataFrame(vifs, columns=["feature","vif"])

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("VIFSelector expects DataFrame.")
        work = X.copy()
        for _ in range(self.max_iter):
            vif_df = self._compute_vif(work)
            cand = vif_df[~vif_df["feature"].isin(self.protected)]
            if len(cand) == 0: break
            row = cand.sort_values("vif", ascending=False).iloc[0]
            vmax = float(row["vif"]); col_remove = str(row["feature"])
            if np.isfinite(vmax) and vmax > self.thr and work.shape[1] >= 2:
                if col_remove in work.columns:
                    work = work.drop(columns=[col_remove])
            else:
                break
        self.keep_columns_ = list(work.columns)
        return self

    def transform(self, X: pd.DataFrame):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.keep_columns_)
        return X[self.keep_columns_].astype(float)

    def get_feature_names_out(self):
        return np.array(self.keep_columns_, dtype=object)

class ImportanceZeroDropSelector(BaseEstimator, TransformerMixin):
    def __init__(self, task: str = "binary", min_estimators: int = 300, random_state: int = 42):
        assert task in ("binary", "reg")
        self.task = task; self.min_estimators = int(min_estimators); self.random_state = int(random_state)
        self.keep_columns_: List[str] = []; self.importances_: pd.DataFrame | None = None

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame): raise TypeError("ImportanceZeroDropSelector expects DataFrame.")
        if y is None: raise ValueError("ImportanceZeroDropSelector requires y.")
        Z = X.copy()
        for c in Z.columns:
            if not np.issubdtype(Z[c].dtype, np.number):
                Z[c] = pd.to_numeric(Z[c], errors="coerce")
            if Z[c].isna().any():
                Z[c] = Z[c].fillna(Z[c].median())
        if self.task == "binary":
            mdl = xgb.XGBClassifier(objective="binary:logistic", n_estimators=max(self.min_estimators, 300),
                                    random_state=self.random_state, **xgb_gpu_params())
        else:
            mdl = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=max(self.min_estimators, 300),
                                   random_state=self.random_state, **xgb_gpu_params())
        mdl.set_params(max_bin=TierFinal.XGB_MAX_BIN, verbosity=0)
        mdl.fit(Z.values.astype(np.float32), np.asarray(y))
        boost_imp = mdl.get_booster().get_score(importance_type="gain") or mdl.get_booster().get_score(importance_type="weight")
        s = pd.Series(boost_imp, dtype=float)
        fmap = {f"f{i}": col for i, col in enumerate(Z.columns)}
        s.index = [fmap.get(k, k) for k in s.index]
        s = s[s > 0.0].sort_values(ascending=False)
        self.keep_columns_ = list(s.index) if len(s) > 0 else list(Z.columns)
        self.importances_ = pd.DataFrame({"feature": s.index, "importance": s.values})
        return self

    def transform(self, X: pd.DataFrame):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.keep_columns_)
        return X[self.keep_columns_].astype(float)

    def get_feature_names_out(self):
        return np.array(self.keep_columns_, dtype=object)

# ───────────────────────────────
# Winsorize/Log
# ───────────────────────────────
class WinsorizeLog1pTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, q_low=0.01, q_high=0.99, skew_thr=1.0,
                 exclude_prefixes=("NIHSS",),
                 exclude_cols=("NIHSS_total_初診時",)):
        self.q_low=q_low; self.q_high=q_high; self.skew_thr=skew_thr
        self.exclude_prefixes=tuple(exclude_prefixes)
        self.exclude_cols=tuple(exclude_cols)
        self.columns_=[]; self.quantiles_={}; self.log_cols_=[]

    def fit(self, X: pd.DataFrame, y=None):
        self.columns_=list(X.columns)
        self.quantiles_.clear(); self.log_cols_.clear()
        for c in self.columns_:
            if c in self.exclude_cols or any(c.startswith(p) for p in self.exclude_prefixes):
                continue
            s=pd.to_numeric(X[c],errors="coerce")
            lo,hi=np.nanpercentile(s,[self.q_low*100,self.q_high*100])
            self.quantiles_[c]=(lo,hi)
            skew_v=s.skew(skipna=True)
            if (np.nanmin(s)>=0) and (abs(skew_v)>self.skew_thr):
                self.log_cols_.append(c)
        return self

    def transform(self,X:pd.DataFrame):
        Z=X[self.columns_].copy()
        for c in self.columns_:
            s=pd.to_numeric(Z[c],errors="coerce")
            if c not in self.exclude_cols and not any(c.startswith(p) for p in self.exclude_prefixes):
                lo,hi=self.quantiles_.get(c,(np.nan,np.nan))
                if np.isfinite(lo) and np.isfinite(hi):
                    s=s.clip(lo,hi)
                if c in self.log_cols_:
                    s=np.log1p(s)
            Z[c]=s.fillna(s.median())
        return Z.astype(np.float32)

    def get_feature_names_out(self):
        return np.array(self.columns_,dtype=object)

# ───────────────────────────────
# FrozenColumnSelector
# ───────────────────────────────
class FrozenColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns: Tuple[str,...]=()):
        self.columns=tuple(columns)
    def fit(self,X,y=None):
        self.columns_=list(self.columns); return self
    def transform(self,X):
        cols=self.columns_
        Xdf = X if isinstance(X,pd.DataFrame) else pd.DataFrame(X,columns=cols)
        return Xdf[cols].astype(float)
    def get_feature_names_out(self):
        return np.array(self.columns_,dtype=object)

# ─────────────────────────────────────────────
# 特徴量定義（log-ratio 方針）
# ─────────────────────────────────────────────
features_initial = [
    '平均気温','平均気圧','Age','Male','BMI',
    'NIHSS_total_初診時',
    't-pa','エダラボン','抗てんかん剤','RAS','Asprin','P2Y12',
    '抗凝固薬','糖尿病治療薬','スタチン','β遮断薬',
    '抗生剤','脳血栓回収術','食事',
    'ad_APTT','ad_HbA1c','ad_LDL_C','ad_PT_INR',
    'ad_Alb','ad_BNP','ad_BUN','ad_CRP','ad_Hb',
    'ad_K','ad_Na','ad_WBC','ad_eGFR',
    'hours_48_心拍数','hours_48_非観血_収縮期',
    'NIHSS_ab質問に対する反応','NIHSS_ac命令への反応','NIHSS_b最良の注視','NIHSS_c視野',
    'NIHSS_d顔面麻痺','NIHSS_e上肢の運動右','NIHSS_f上肢の運動左','NIHSS_g下肢の運動右',
    'NIHSS_h下肢の運動左','NIHSS_i四肢の運動失調','NIHSS_j感覚','NIHSS_k言語',
    'NIHSS_l構音障害','NIHSS_m消去無視',
    'TIA','アテローム血栓性梗塞','その他の脳梗塞','ラクナ梗塞','心原性脳塞栓','脳出血'
]
DELTA_TARGETS = ['Alb','BNP','BUN','CRP','Hb','K','Na','WBC','eGFR']

def _safe_log_ratio(a: pd.Series, b: pd.Series) -> pd.Series:
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    qa = np.nanpercentile(a, 1)
    qb = np.nanpercentile(b, 1)
    eps_a = 1e-6 if (not np.isfinite(qa) or qa <= 0) else float(qa) * 0.5
    eps_b = 1e-6 if (not np.isfinite(qb) or qb <= 0) else float(qb) * 0.5
    ratio = (a + eps_a) / (b + eps_b)
    ratio = ratio.where(ratio > 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.log(ratio.astype(float))
    return out

def build_features_no_impute(df: pd.DataFrame, keep_meta: bool = True) -> pd.DataFrame:
    out = df.copy()
    created = []
    for col in DELTA_TARGETS:
        ad_col = f"ad_{col}"; d3_col = f"hours_48_{col}"
        if ad_col in out.columns and d3_col in out.columns:
            out[f"log_ratio_{col}"] = _safe_log_ratio(out[d3_col], out[ad_col]).clip(-5, 5)
            created.append(f"log_ratio_{col}")
    drop_cols = [c for c in out.columns if c.startswith("hours_48_")
                 and c not in ("hours_48_心拍数", "hours_48_非観血_収縮期")]
    out.drop(columns=drop_cols, inplace=True, errors='ignore')
    base_cols = [c for c in features_initial if c in out.columns]
    use_cols = base_cols + [c for c in created if c in out.columns]
    dt = out[use_cols].copy()
    if keep_meta:
        for mc in [Config.TARGET_COL, Config.LOS_COL, Config.INDEX_COL, Config.TIME_FLAG_COL]:
            if mc in out.columns and mc not in dt.columns:
                dt[mc] = out[mc]
    for c in dt.columns:
        if dt[c].dtype == bool:
            dt[c] = dt[c].astype(int)
    return dt

# ─────────────────────────────────────────────
# 多重代入
# ─────────────────────────────────────────────
META_COLS = [Config.TARGET_COL, Config.LOS_COL, Config.INDEX_COL, Config.TIME_FLAG_COL]

class MIConf:
    M = TierTune.MI_M
    MAX_ITER = TierTune.MI_MAX_ITER
    SAMPLE_POSTERIOR = True
    SEED = 20250815
    USE_STACK_TUNING = True

def _iterative_imputer(seed: int) -> IterativeImputer:
    return IterativeImputer(
        random_state=seed,
        max_iter=MIConf.MAX_ITER,
        sample_posterior=MIConf.SAMPLE_POSTERIOR,
        skip_complete=True,
        tol=1e-3,
        imputation_order="roman",
        n_nearest_features=20,
        min_value=None, max_value=None,
        initial_strategy="median"
    )

def multiple_impute_then_build(raw_df: pd.DataFrame,
                               feature_builder_fn,
                               features_needed: List[str]) -> List[pd.DataFrame]:
    df = raw_df.copy()
    if Config.INDEX_COL not in df.columns:
        logger.warning(f"'{Config.INDEX_COL}' が無いため 0..N-1 を付与。")
        df[Config.INDEX_COL] = np.arange(len(df), dtype=int)
    if Config.TARGET_COL not in df.columns:
        raise RuntimeError(f"'{Config.TARGET_COL}' がありません。")
    df = df[~df[Config.TARGET_COL].isna()].copy()
    if Config.TIME_FLAG_COL not in df.columns:
        raise RuntimeError(f"時間検証カラム '{Config.TIME_FLAG_COL}' がありません（0/1必須）。")

    core_cols = [c for c in features_needed if c in df.columns]
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in META_COLS]
    impute_cols = sorted(set(core_cols + num_cols))

    msk_dev = (df[Config.TIME_FLAG_COL] == 0)
    imputed_list: List[pd.DataFrame] = []
    rng = np.random.default_rng(MIConf.SEED)

    for _ in range(MIConf.M):
        seed = int(rng.integers(1, 10_000_000))
        imp = _iterative_imputer(seed)

        Xd = df.loc[msk_dev,  impute_cols].astype(float)
        Xh = df.loc[~msk_dev, impute_cols].astype(float)

        imp.fit(Xd)
        Xd_imp = imp.transform(Xd)
        Xh_imp = imp.transform(Xh)

        imputed = df.copy()
        imputed.loc[msk_dev,  impute_cols] = Xd_imp
        imputed.loc[~msk_dev, impute_cols] = Xh_imp

        dt = feature_builder_fn(imputed, keep_meta=True)
        dt[Config.TARGET_COL] = pd.to_numeric(dt[Config.TARGET_COL], errors="coerce").astype(int)
        dt[Config.TARGET_COL] = (dt[Config.TARGET_COL] != 0).astype(int)

        if Config.LOS_COL in dt.columns:
            dt[Config.LOS_COL] = pd.to_numeric(dt[Config.LOS_COL], errors="coerce").clip(upper=Config.LOS_WINSOR_MAX)
        if Config.LOS_COL in dt.columns:
            dt[Config.LOS_BIN_COL] = (dt[Config.LOS_COL] > Config.LOS_CUTOFF_DAYS).astype(int)

        imputed_list.append(dt)

    return imputed_list

# ─────────────────────────────────────────────
# 強制保持
# ─────────────────────────────────────────────
TASK_NAMES = ["OUTCOME", "LOSBIN", "LOSREG"]

# ★ 未定義エラー対策：環境変数 or None
FORCE_MAP_CSV_PATH = os.getenv("FORCE_MAP_CSV_PATH", None)

def load_force_map(csv_path: str | None, fallback_global: List[str]) -> Dict[str, List[str]]:
    force: Dict[str, List[str]] = {t: list(fallback_global) for t in TASK_NAMES}
    if not csv_path:
        logger.info("[FORCE] CSV未指定: グローバル強制保持のみ")
        return force
    try:
        df = pd.read_csv(csv_path)
        need = {"Task","Feature","Forced"}
        if not need.issubset(df.columns):
            raise ValueError("CSVに Task, Feature, Forced 列が必要です。")
        df["Task"] = df["Task"].astype(str)
        df["Feature"] = df["Feature"].astype(str)
        df["Forced"] = pd.to_numeric(df["Forced"], errors="coerce").fillna(0).astype(int)
        for t in TASK_NAMES:
            feat = df.query("Task == @t and Forced == 1")["Feature"].unique().tolist()
            force[t] = sorted(set(force[t]).union(set(feat)))
        logger.info("[FORCE] CSVを読み込み、タスク別の強制保持を設定しました。")
    except Exception as e:
        logger.warning(f"[FORCE] CSV読み込み失敗: グローバルのみ適用 detail={e}")
    return force

FORCE_BY_TASK = load_force_map(FORCE_MAP_CSV_PATH, GLOBAL_FORCE_KEEP_COLS)

def learn_heavy_selection_once(dt_list: List[pd.DataFrame],
                               label_col: str,
                               vif_thr: float,
                               task: str,
                               force_keep: List[str] | None = None) -> List[str]:
    X_stack, y_stack, _ = stack_for_tuning(dt_list, label_col=label_col)
    protected = [c for c in (force_keep or []) if c in X_stack.columns]
    pre_heavy = Pipeline([
        ("vif", VIFSelector(thr=vif_thr, max_iter=30, add_constant=True, protected=protected)),
        ("imp0drop", ImportanceZeroDropSelector(task=("binary" if task=="binary" else "reg"),
                                               min_estimators=300, random_state=Config.RANDOM_STATE))
    ])
    pre_heavy.fit(X_stack, y_stack)
    keep = list(pre_heavy.named_steps["imp0drop"].get_feature_names_out())
    if protected:
        keep = sorted(set(keep).union(set(protected)))
    return keep

def make_runtime_preprocess_from_keep(keep_columns: List[str]) -> Pipeline:
    return Pipeline([("keep", FrozenColumnSelector(tuple(keep_columns))),
                     ("winsor_log", WinsorizeLog1pTransformer(
                         exclude_prefixes=("NIHSS",),
                         exclude_cols=("NIHSS_total_初診時",)
                     ))])

# ─────────────────────────────────────────────
# CV / 補助
# ─────────────────────────────────────────────
def make_sample_weight(y: np.ndarray) -> np.ndarray:
    classes = np.unique(y)
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    wmap = {c: w for c, w in zip(classes, cw)}
    return np.array([wmap[v] for v in y], dtype=float)

def prepare_from_dt_timeaware(dt_with_meta: pd.DataFrame, label_col: str) -> Tuple[pd.DataFrame, np.ndarray, pd.DataFrame, np.ndarray]:
    for col in [label_col, Config.TIME_FLAG_COL]:
        if col not in dt_with_meta.columns:
            raise RuntimeError(f"dt_with_meta に '{col}' がありません。")
    if label_col != Config.LOS_COL:
        y = pd.to_numeric(dt_with_meta[label_col], errors="coerce").astype(int).to_numpy()
    else:
        y = dt_with_meta[label_col].astype(float).to_numpy()
    meta_cols = [c for c in [Config.TARGET_COL, Config.LOS_COL, Config.LOS_BIN_COL, Config.INDEX_COL, Config.TIME_FLAG_COL]
                 if c in dt_with_meta.columns]
    X = dt_with_meta.drop(columns=meta_cols)
    msk_hold = (dt_with_meta[Config.TIME_FLAG_COL] == 1).to_numpy()
    dev_X, hold_X = X.loc[~msk_hold], X.loc[msk_hold]
    dev_y, hold_y = y[~msk_hold], y[msk_hold]
    logger.info(f"Temporal split[{label_col}]: dev={len(dev_X)}, hold={len(hold_X)}")
    return dev_X, dev_y, hold_X, hold_y

def stack_for_tuning(dt_list: List[pd.DataFrame], label_col: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    Xs, ys, gs = [], [], []
    for dt in dt_list:
        Xd, yd, _, _ = prepare_from_dt_timeaware(dt, label_col=label_col)
        Xs.append(Xd); ys.append(yd)
        gs.append(dt.loc[Xd.index, Config.INDEX_COL].values)
    return pd.concat(Xs, axis=0), np.concatenate(ys, axis=0), np.concatenate(gs, axis=0)

def split_indices_from_first(dt_list: List[pd.DataFrame]) -> Tuple[np.ndarray, np.ndarray]:
    dt0 = dt_list[0]
    assert Config.INDEX_COL in dt0.columns
    msk_hold = (dt0[Config.TIME_FLAG_COL] == 1).to_numpy()
    idx_all = dt0.index.to_numpy()
    return idx_all[~msk_hold], idx_all[msk_hold]

def _build_cv_for_classification(y: np.ndarray, groups: np.ndarray | None, n_splits: int):
    if groups is not None:
        uniq = np.unique(groups)
        if len(uniq) < len(y) * 0.9:
            return GroupKFold(n_splits=n_splits)
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=Config.RANDOM_STATE)

def _build_cv_for_regression(groups: np.ndarray | None, n_splits: int):
    if groups is not None:
        uniq = np.unique(groups)
        if len(uniq) < len(groups) * 0.9:
            return GroupKFold(n_splits=n_splits)
    return KFold(n_splits=n_splits, shuffle=True, random_state=Config.RANDOM_STATE)

def rmse_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    try:
        return float(mean_squared_error(y_true, y_pred, squared=False))
    except TypeError:
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def calc_calibration_slope_citl(y_true: np.ndarray, y_prob1: np.ndarray) -> Tuple[float, float]:
    from sklearn.linear_model import LogisticRegression as LR
    p = np.clip(y_prob1, 1e-6, 1 - 1e-6)
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    y = y_true.astype(int).ravel()
    try:
        lr = LR(C=1e6, solver="lbfgs", max_iter=1000)
        lr.fit(logit, y)
        slope = float(lr.coef_.ravel()[0])
        citl  = float(lr.intercept_.ravel()[0])
        return slope, citl
    except Exception:
        return float("nan"), float("nan")

def get_bootstrap_ci(y_true: np.ndarray, y_prob: np.ndarray, n_boot: int):
    rng = np.random.default_rng(Config.RANDOM_STATE)
    n = len(y_true)
    aucs, aps, briers = [], [], []
    slopes, citls = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yy = y_true[idx]; pp = y_prob[idx]
        try: aucs.append(roc_auc_score(yy, pp[:,1]))
        except: aucs.append(np.nan)
        try: aps.append(average_precision_score(yy, pp[:,1]))
        except: aps.append(np.nan)
        try: briers.append(float(np.mean((pp[:,1]-yy)**2)))
        except: briers.append(np.nan)
        try:
            s, c = calc_calibration_slope_citl(yy, pp[:,1])
            slopes.append(s); citls.append(c)
        except:
            slopes.append(np.nan); citls.append(np.nan)
    def ci(xs):
        xs = np.array(xs, float); pt = float(np.nanmean(xs))
        lo = float(np.nanpercentile(xs, 2.5)); hi = float(np.nanpercentile(xs, 97.5))
        return pt, lo, hi
    return {
        "AUROC": ci(aucs), "AP": ci(aps), "Brier": ci(briers),
        "Slope": ci(slopes), "CITL": ci(citls)
    }

# ─────────────────────────────────────────────
# 学習器（Stage別探索空間）
# ─────────────────────────────────────────────
def build_estimator(name: str, stage: str):
    if name == "Logistic":
        mdl = LogisticRegression(max_iter=4000, random_state=Config.RANDOM_STATE, solver="saga", n_jobs=-1)
        if stage == "TUNE":
            space = {
                "model__C": optuna.distributions.FloatDistribution(5e-4, 5e0, log=True),
                "model__penalty": optuna.distributions.CategoricalDistribution(choices=["l2","l1","elasticnet"]),
                "model__l1_ratio": optuna.distributions.FloatDistribution(0.0, 1.0)
            }
        elif stage == "FINAL":
            space = {
                "model__C": optuna.distributions.FloatDistribution(1e-4, 1e1, log=True),
                "model__penalty": optuna.distributions.CategoricalDistribution(choices=["elasticnet","l2"]),
                "model__l1_ratio": optuna.distributions.CategoricalDistribution(choices=[0.2, 0.5, 0.8])
            }
        else:  # QUICK
            space = {
                "model__C": optuna.distributions.FloatDistribution(1e-3, 1e0, log=True),
                "model__penalty": optuna.distributions.CategoricalDistribution(choices=["l2","elasticnet"]),
                "model__l1_ratio": optuna.distributions.CategoricalDistribution(choices=[0.5])
            }
        return mdl, space

    elif name == "XGBoost":
        if stage == "TUNE":
            n_est = TierTune.XGB_N_EST_CLS; max_bin = TierTune.XGB_MAX_BIN; lr_low = 0.02
        elif stage == "FINAL":
            n_est = TierFinal.XGB_N_EST_CLS; max_bin = TierFinal.XGB_MAX_BIN; lr_low = 0.01
        else:  # QUICK
            n_est = TierQuick.XGB_N_EST_CLS; max_bin = TierQuick.XGB_MAX_BIN; lr_low = 0.03
        base_kwargs = dict(objective="binary:logistic", eval_metric="logloss",
                           n_estimators=n_est, random_state=Config.RANDOM_STATE, **xgb_gpu_params())
        mdl = xgb.XGBClassifier(**base_kwargs)
        mdl.set_params(max_bin=max_bin, n_jobs=-1)
        space = {
            "model__max_depth":            optuna.distributions.IntDistribution(3, 8),
            "model__learning_rate":        optuna.distributions.FloatDistribution(lr_low, 5e-2, log=True),
            "model__min_child_weight":     optuna.distributions.IntDistribution(1, 10),
            "model__gamma":                optuna.distributions.FloatDistribution(0, 4),
            "model__subsample":            optuna.distributions.FloatDistribution(.6, .9),
            "model__colsample_bytree":     optuna.distributions.FloatDistribution(.6, .9),
            "model__reg_alpha":            optuna.distributions.FloatDistribution(1e-4, 1, log=True),
            "model__reg_lambda":           optuna.distributions.FloatDistribution(1e-3, 5, log=True)
        }
        return mdl, space

    else:
        raise ValueError(name)

def build_regressor(stage: str):
    if stage == "TUNE":
        n_est = TierTune.XGB_N_EST_REG; max_bin = TierTune.XGB_MAX_BIN
    elif stage == "FINAL":
        n_est = TierFinal.XGB_N_EST_REG; max_bin = TierFinal.XGB_MAX_BIN
    else:  # QUICK
        n_est = TierQuick.XGB_N_EST_REG; max_bin = TierQuick.XGB_MAX_BIN
    reg = xgb.XGBRegressor(objective="reg:squarederror",
                           n_estimators=n_est,
                           random_state=Config.RANDOM_STATE, **xgb_gpu_params())
    reg.set_params(max_bin=max_bin, n_jobs=-1)
    space = {
        "model__max_depth":        optuna.distributions.IntDistribution(3, 10),
        "model__learning_rate":    optuna.distributions.FloatDistribution(1e-3, 5e-2, log=True),
        "model__min_child_weight": optuna.distributions.IntDistribution(1, 15),
        "model__gamma":            optuna.distributions.FloatDistribution(0, 5),
        "model__subsample":        optuna.distributions.FloatDistribution(.6, .95),
        "model__colsample_bytree": optuna.distributions.FloatDistribution(.6, .95),
        "model__reg_alpha":        optuna.distributions.FloatDistribution(1e-4, 10, log=True),
        "model__reg_lambda":       optuna.distributions.FloatDistribution(1e-4, 10, log=True)
    }
    return reg, space

# ─────────────────────────────────────────────
# Optuna（枝刈り）
# ─────────────────────────────────────────────
def tune_hyperparameters_for_label(stage: str, label_col: str, base_model, space, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, es_rounds: int, n_splits: int, n_trials: int, patience: int):
    keep = KEEP_OUTCOME if label_col == Config.TARGET_COL else KEEP_LOSBIN
    pre = make_runtime_preprocess_from_keep(keep)
    pipe = Pipeline([("pre", pre), ("model", base_model)])
    cv = _build_cv_for_classification(y, groups, n_splits)

    def objective(trial: optuna.Trial) -> float:
        params = {}
        for k, d in space.items():
            if isinstance(d, optuna.distributions.FloatDistribution):
                params[k] = trial.suggest_float(k, d.low, d.high, log=d.log)
            elif isinstance(d, optuna.distributions.IntDistribution):
                params[k] = trial.suggest_int(k, d.low, d.high, step=d.step)
            else:
                params[k] = trial.suggest_categorical(k, d.choices)
        if params.get("model__penalty", "l2") != "elasticnet":
            params.pop("model__l1_ratio", None)

        pipe.set_params(**params)
        oof = np.zeros((len(y), 2), dtype=np.float32)

        for fold_id, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups if isinstance(cv, GroupKFold) else None)):
            Xtr, ytr = X.iloc[tr_idx], y[tr_idx]
            Xva, yva = X.iloc[va_idx], y[va_idx]
            pre_f = clone(pre).fit(Xtr, ytr)
            Xtr_t = pre_f.transform(Xtr); Xva_t = pre_f.transform(Xva)

            model = clone(base_model)
            sw = make_sample_weight(ytr)
            eskw = _build_es_kwargs_model(model, Xva_t, yva, es_rounds)
            if isinstance(model, xgb.XGBClassifier):
                model_es, _ = fit_with_gpu_fallback_xgb(model, Xtr_t, ytr, sample_weight=sw, fit_kwargs=eskw)
            else:
                model.fit(Xtr_t, ytr, sample_weight=sw)
                model_es = model
            try:
                proba = model_es.predict_proba(Xva_t)
            except Exception:
                from scipy.special import expit
                s = model_es.decision_function(Xva_t)
                proba = np.vstack([1-expit(s), expit(s)]).T
            oof[va_idx] = proba

            auc_partial = roc_auc_score(y[:va_idx.max()+1], oof[:va_idx.max()+1,1])
            trial.report(auc_partial, step=fold_id)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return roc_auc_score(y, oof[:,1])

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=Config.RANDOM_STATE, multivariate=True, group=True),
                                pruner=optuna.pruners.HyperbandPruner(min_resource=1, reduction_factor=3))
    def _early_stop_cb(study_, trial_):
        if len(study_.trials) - study_.best_trial.number > patience:
            study_.stop()

    study.optimize(objective, n_trials=n_trials, callbacks=[_early_stop_cb], show_progress_bar=False)
    best_params = {k.replace("model__", ""): v for k, v in study.best_params.items()}
    return best_params

def tune_for_label(stage: str, label_col: str, dt_list: List[pd.DataFrame], n_splits: int, n_trials: int, patience: int, es_rounds: int):
    X_stack, y_stack, g_stack = stack_for_tuning(dt_list, label_col=label_col)
    tuned_params_for_label = {}
    for mname in ["XGBoost", "Logistic"]:
        base_model, space = build_estimator(mname, stage)
        best_params = tune_hyperparameters_for_label(stage, label_col, base_model, space, X_stack, y_stack, g_stack, es_rounds, n_splits, n_trials, patience)
        tuned_params_for_label[mname] = best_params
        logger.info(f"[{stage}-TUNED - {label_col} - {mname}] {best_params}")
    return tuned_params_for_label

# ─────────────────────────────────────────────
# 回帰チューニング
# ─────────────────────────────────────────────
def cv_predict_regression_group(pipe: Pipeline, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, n_splits:int) -> np.ndarray:
    cv = _build_cv_for_regression(groups, n_splits)
    oof = np.zeros(len(y), dtype=float)
    splitter = cv.split(X, y, groups if isinstance(cv, GroupKFold) else None)
    for tr_idx, va_idx in splitter:
        Xtr, ytr = X.iloc[tr_idx], y[tr_idx]
        Xva = X.iloc[va_idx]
        ytr_log = np.log1p(ytr)
        p = clone(pipe)
        p.fit(Xtr, ytr_log)
        pred_log = p.predict(Xva)
        oof[va_idx] = np.expm1(pred_log).clip(0, None)
    return oof

def tune_regression(stage: str, dt_list: List[pd.DataFrame], n_splits:int, n_trials:int, patience:int):
    Xreg_stack, yreg_stack, greg_stack = stack_for_tuning(dt_list, label_col=Config.LOS_COL)
    keep = KEEP_LOSREG
    pre_reg = make_runtime_preprocess_from_keep(keep)
    reg_xgb_base, reg_xgb_space = build_regressor(stage)
    pipe_reg_xgb = Pipeline([("pre", pre_reg), ("model", reg_xgb_base)])
    def objective(trial: optuna.Trial) -> float:
        params = {}
        for k, d in reg_xgb_space.items():
            if isinstance(d, optuna.distributions.FloatDistribution):
                params[k] = trial.suggest_float(k, d.low, d.high, log=d.log)
            elif isinstance(d, optuna.distributions.IntDistribution):
                params[k] = trial.suggest_int(k, d.low, d.high, step=d.step)
            else:
                params[k] = trial.suggest_categorical(k, d.choices)
        pipe_reg_xgb.set_params(**params)
        oof = cv_predict_regression_group(pipe_reg_xgb, Xreg_stack, yreg_stack, greg_stack, n_splits)
        rmse = rmse_score(yreg_stack, oof)
        trial.report(-rmse, step=0)
        return -rmse
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=Config.RANDOM_STATE, multivariate=True, group=True),
                                pruner=optuna.pruners.HyperbandPruner(min_resource=1, reduction_factor=3))
    def _early_stop_cb(study_, trial_):
        if len(study_.trials) - study_.best_trial.number > patience:
            study_.stop()
    study.optimize(objective, n_trials=n_trials, callbacks=[_early_stop_cb], show_progress_bar=False)
    best_params = {k.replace("model__", ""): v for k, v in study.best_params.items()}
    pipe_reg_xgb.set_params(**study.best_params)
    return pipe_reg_xgb, best_params

# ─────────────────────────────────────────────
# 単一校正器
# ─────────────────────────────────────────────
class ProbCalibrator:
    def __init__(self, method: str = "sigmoid"):
        assert method in ("sigmoid", "isotonic", "none")
        self.method = method
        self.lr_: LogisticRegression | None = None
        self.iso_: IsotonicRegression | None = None

    def fit(self, y_true: np.ndarray, p1: np.ndarray):
        y = y_true.astype(int).ravel()
        p1 = np.clip(p1.astype(float).ravel(), 1e-6, 1-1e-6)
        if self.method == "sigmoid":
            x = np.log(p1 / (1 - p1)).reshape(-1, 1)
            lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
            lr.fit(x, y)
            self.lr_ = lr
            self.iso_ = None
        elif self.method == "isotonic":
            iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
            iso.fit(p1, y)
            self.iso_ = iso
            self.lr_ = None
        else:
            self.lr_ = None; self.iso_ = None
        return self

    def transform(self, p1: np.ndarray) -> np.ndarray:
        if self.method == "none":
            return np.clip(p1.astype(float).ravel(), 0, 1)
        p1 = np.clip(p1.astype(float).ravel(), 1e-6, 1-1e-6)
        if self.lr_ is not None:
            x = np.log(p1 / (1 - p1)).reshape(-1, 1)
            return self.lr_.predict_proba(x)[:,1]
        elif self.iso_ is not None:
            return np.clip(self.iso_.transform(p1), 0.0, 1.0)
        else:
            return p1

# ─────────────────────────────────────────────
# OOF生成 & HOLD予測
# ─────────────────────────────────────────────
def oof_predict_binary(base_model, keep_cols: List[str], Xd: pd.DataFrame, yd: np.ndarray,
                       n_splits: int, es_rounds: int, model_name: str) -> Tuple[np.ndarray, Pipeline]:
    pre_template = make_runtime_preprocess_from_keep(keep_cols)
    cv = _build_cv_for_classification(yd, None, n_splits)
    oof = np.zeros((len(yd), 2), dtype=np.float32)
    for tr_idx, va_idx in cv.split(Xd, yd):
        Xtr, ytr = Xd.iloc[tr_idx], yd[tr_idx]
        Xva, yva = Xd.iloc[va_idx], yd[va_idx]
        pre_f = clone(pre_template).fit(Xtr, ytr)
        Xtr_t = pre_f.transform(Xtr); Xva_t = pre_f.transform(Xva)

        mdl = clone(base_model)
        sw = make_sample_weight(ytr)
        eskw = _build_es_kwargs_model(mdl, Xva_t, yva, es_rounds)
        if isinstance(mdl, xgb.XGBClassifier):
            mdl_es, _ = fit_with_gpu_fallback_xgb(mdl, Xtr_t, ytr, sample_weight=sw, fit_kwargs=eskw)
        else:
            mdl.fit(Xtr_t, ytr, sample_weight=sw)
            mdl_es = mdl
        try:
            proba = mdl_es.predict_proba(Xva_t)
        except Exception:
            from scipy.special import expit
            s = mdl_es.decision_function(Xva_t)
            proba = np.vstack([1-expit(s), expit(s)]).T
        oof[va_idx] = proba
    return oof, pre_template

def fit_full_and_predict_hold(base_model, pre_template: Pipeline, Xd: pd.DataFrame, yd: np.ndarray,
                              Xh: pd.DataFrame, es_rounds: int, task_tag: str, model_tag: str,
                              feat_names: List[str], idx_hold: np.ndarray, mi_id: int):
    # 内部 val で ES
    Xtr, Xev, ytr, yev = train_test_split(Xd, yd, test_size=0.15, random_state=Config.RANDOM_STATE, stratify=yd)
    pre = clone(pre_template).fit(Xtr, ytr)
    Xtr_t = pre.transform(Xtr); Xev_t = pre.transform(Xev)
    Xh_t  = pre.transform(Xh)

    mdl = clone(base_model)
    sw = make_sample_weight(ytr)
    eskw = _build_es_kwargs_model(mdl, Xev_t, yev, es_rounds)
    if isinstance(mdl, xgb.XGBClassifier):
        mdl_es, _ = fit_with_gpu_fallback_xgb(mdl, Xtr_t, ytr, sample_weight=sw, fit_kwargs=eskw)
    else:
        mdl.fit(Xtr_t, ytr, sample_weight=sw)
        mdl_es = mdl

    try:
        proba_hold = mdl_es.predict_proba(Xh_t)
    except Exception:
        from scipy.special import expit
        s = mdl_es.decision_function(Xh_t)
        proba_hold = np.vstack([1-expit(s), expit(s)]).T

    save_model_bundle(task_tag, model_tag, "raw_full_dev", mdl_es, pre, Xh_t, idx_hold, feat_names, mi_id=mi_id)
    return proba_hold

# ─────────────────────────────────────────────
# 保存ユーティリティ（SHAP/feature名ズレ対策版）
# ─────────────────────────────────────────────
PRED_DIR = Config.RESULT_DIR / Config.PRED_DIRNAME
ARTIFACT_DIR = Config.RESULT_DIR / Config.ARTIFACT_DIRNAME
PRED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def _resolve_feature_names_from_pre(pre, fallback=None) -> List[str]:
    """
    前処理パイプラインから列名を取得。失敗時は fallback を返す。
    """
    try:
        if hasattr(pre, "get_feature_names_out"):
            names = pre.get_feature_names_out()
            return [str(x) for x in names]
    except Exception as e:
        logger.debug(f"[ARTIFACT] get_feature_names_out failed: {e}")
    return [str(x) for x in (fallback or [])]

def save_model_bundle(
    task: str,
    mname: str,
    calib: str,
    model,
    pre,
    X_hold_t,                 # ndarray でも DataFrame でも可
    index_hold,               # 1D
    feature_names,            # フォールバック用
    mi_id: int | None = None
):
    """
    artifact に SHAP 用の情報を一貫した形で保存する。
    - feature_names は以下の優先度で決定:
        1) X_hold_t が DataFrame なら X_hold_t.columns
        2) pre.get_feature_names_out()
        3) 引数 feature_names
    - X_hold_t は最終的に ndarray(float) として保存
    """
    # 1) X_hold_t が DataFrame なら列名を優先
    feat_from_df = None
    X_arr = X_hold_t
    try:
        if isinstance(X_hold_t, pd.DataFrame):
            feat_from_df = [str(c) for c in X_hold_t.columns]
            X_arr = X_hold_t.values
    except Exception:
        pass

    # 2) pre から列名
    feat_from_pre = _resolve_feature_names_from_pre(pre, fallback=None)

    # 3) 引数由来のフォールバック（Indexでも安全に処理）
    if feature_names is None:
        feat_from_arg = []
    else:
        try:
            feat_iterable = feature_names.tolist() if hasattr(feature_names, "tolist") else feature_names
            feat_from_arg = [str(x) for x in feat_iterable]
        except Exception:
            feat_from_arg = [str(x) for x in list(feature_names)]

    # 優先順位で決定
    if feat_from_df and len(feat_from_df) > 0:
        final_feat_names = feat_from_df
    elif feat_from_pre and len(feat_from_pre) > 0:
        final_feat_names = feat_from_pre
    else:
        final_feat_names = feat_from_arg

    # 形状チェック：列数と名前数がズレていれば警告（保存は強行）
    try:
        if X_arr is not None and hasattr(X_arr, "shape"):
            n_feat = X_arr.shape[1]
            if len(final_feat_names) != n_feat:
                logger.warning(
                    f"[ARTIFACT] feature_names length mismatch: "
                    f"{len(final_feat_names)} (names) vs {n_feat} (X_hold_t.shape[1]). "
                    f"Proceeding with current names."
                )
    except Exception:
        pass

    # 型整形
    X_arr = np.array(X_arr, dtype=float)
    index_arr = np.array(index_hold)

    bundle = dict(
        model=model,
        pre=pre,
        feature_names=list(final_feat_names),
        X_hold_t=X_arr,
        index_hold=index_arr,
    )

    mi_tag = f"__MI{mi_id}" if mi_id is not None else ""
    fname = ARTIFACT_DIR / f"{task}__{mname}__{calib}{mi_tag}.pkl"
    joblib.dump(bundle, fname, compress=3)
    logger.info(f"[ARTIFACT] Saved → {fname.name} (feat={len(final_feat_names)}, holdN={len(index_arr)})")

def save_calibrator_bundle(task: str, mname: str, method: str, calibrator: ProbCalibrator):
    fname = ARTIFACT_DIR / f"{task}__{mname}__calibrator__{method}.pkl"
    joblib.dump(calibrator, fname, compress=3)
    logger.info(f"[ARTIFACT] Saved → {fname.name}")

def save_predictions_csv(task: str, mname: str, calib: str,
                         index_hold: np.ndarray, y_true: np.ndarray, y_prob1: np.ndarray):
    df = pd.DataFrame({
        "INDEX": index_hold,
        "y_true": y_true.astype(int),
        "y_prob": y_prob1.astype(float)
    })
    out = PRED_DIR / f"{task}__{mname}__{calib}__hold.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    logger.info(f"[PRED] Saved → {out.name}")

def save_los_predictions_named(task_tag: str, model_tag: str,
                               index_hold: np.ndarray, y_true_los: np.ndarray, y_pred_los: np.ndarray):
    df = pd.DataFrame({
        "INDEX": index_hold,
        "LOS_true": y_true_los.astype(float),
        "LOS_pred": y_pred_los.astype(float)
    })
    out = PRED_DIR / f"{task_tag}__{model_tag}__raw__hold.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")
    logger.info(f"[PRED] Saved → {out.name}")

def plot_calibration_curve_binary(y_true: np.ndarray, y_prob1: np.ndarray, fname: Path, title: str, label: str):
    prob_true, prob_pred = calibration_curve(y_true, y_prob1, n_bins=Config.CAL_N_BINS, strategy='quantile')
    plt.figure(figsize=(5.8, 5.5))
    plt.plot([0, 1], [0, 1], "--", label="Perfect")
    plt.plot(prob_pred, prob_true, marker="o", linewidth=1.8, label=label)
    plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency")
    plt.title(title); plt.legend(loc="upper left")
    plt.tight_layout(); plt.savefig(fname, dpi=600); plt.close()

def save_confusion_matrix_plot(cm: np.ndarray, class_names: List[str], model_name: str, prefix: str):
    plt.figure(figsize=(7.2, 6))
    plt.imshow(cm, interpolation='nearest')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, f"{cm[i,j]}", ha="center", va="center")
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(Config.RESULT_DIR / f"{prefix}_{model_name}_cm.png", dpi=600, bbox_inches='tight')
    plt.close()

# ─────────────────────────────────────────────
# 実行フロー（TUNE → FINAL/QUICK）
# ─────────────────────────────────────────────
def run_stage(stage: str, raw: pd.DataFrame):
    global dt_list_global
    if stage == "TUNE":
        MIConf.M = TierTune.MI_M
        MIConf.MAX_ITER = TierTune.MI_MAX_ITER
        N_SPLITS = TierTune.N_SPLITS
        N_TRIALS = TierTune.N_TRIALS
        PATIENCE = TierTune.PATIENCE
        ES_ROUNDS = TierTune.ES_ROUNDS
        N_BOOT = TierTune.N_BOOT
    elif stage == "FINAL":
        MIConf.M = TierFinal.MI_M
        MIConf.MAX_ITER = TierFinal.MI_MAX_ITER
        N_SPLITS = TierFinal.N_SPLITS
        N_TRIALS = TierFinal.N_TRIALS
        PATIENCE = TierFinal.PATIENCE
        ES_ROUNDS = TierFinal.ES_ROUNDS
        N_BOOT = TierFinal.N_BOOT
    elif stage == "QUICK":
        MIConf.M = TierQuick.MI_M
        MIConf.MAX_ITER = TierQuick.MI_MAX_ITER
        N_SPLITS = TierQuick.N_SPLITS
        N_TRIALS = TierQuick.N_TRIALS
        PATIENCE = TierQuick.PATIENCE
        ES_ROUNDS = TierQuick.ES_ROUNDS
        N_BOOT = TierQuick.N_BOOT
    else:
        raise ValueError(f"Unknown stage: {stage}")

    logger.info(f"=== RUN STAGE: {stage} ===")
    dt_list = multiple_impute_then_build(
        raw_df=raw,
        feature_builder_fn=build_features_no_impute,
        features_needed=features_initial + [f"ad_{c}" for c in DELTA_TARGETS] + [f"hours_48_{c}" for c in DELTA_TARGETS]
    )
    dt_list_global = dt_list
    dt_preview = dt_list[0].copy()
    dt_preview.to_csv(Config.RESULT_DIR / f"dt_with_meta_preview_MI1__{stage}.csv", index=True, encoding="utf-8-sig")

    # 特徴選択（重め）は各タスク1回だけ
    global KEEP_OUTCOME, KEEP_LOSBIN, KEEP_LOSREG
    KEEP_OUTCOME = learn_heavy_selection_once(dt_list, label_col=Config.TARGET_COL,  vif_thr=8.0, task="binary", force_keep=FORCE_BY_TASK["OUTCOME"])
    KEEP_LOSBIN  = learn_heavy_selection_once(dt_list, label_col=Config.LOS_BIN_COL, vif_thr=8.0, task="binary", force_keep=FORCE_BY_TASK["LOSBIN"])
    KEEP_LOSREG  = learn_heavy_selection_once(dt_list, label_col=Config.LOS_COL,     vif_thr=8.0, task="reg",    force_keep=FORCE_BY_TASK["LOSREG"])

    # ハイパラ探索
    BEST_PARAMS_OUTCOME = tune_for_label(stage, Config.TARGET_COL, dt_list, N_SPLITS, N_TRIALS, PATIENCE, ES_ROUNDS)
    BEST_PARAMS_LOS     = tune_for_label(stage, Config.LOS_BIN_COL, dt_list, N_SPLITS, N_TRIALS, PATIENCE, ES_ROUNDS)
    pipe_reg, BEST_PARAMS_REG_XGB = tune_regression(stage, dt_list, N_SPLITS, N_TRIALS, PATIENCE)
    logger.info(f"[{stage}] BEST_PARAMS_REG_XGB: {BEST_PARAMS_REG_XGB}")

    # 成果物生成（FINAL と QUICK で実行）
    if stage in ("FINAL", "QUICK"):
        device_records: List[str] = []
        dev_idx, hold_idx = split_indices_from_first(dt_list)

        # OUTCOME / LOSBIN の OOF と HOLD raw を貯める器
        oof_dev_outcome: Dict[str, List[pd.DataFrame]] = {"XGBoost": [], "Logistic": []}
        hold_raw_outcome: Dict[str, List[pd.DataFrame]] = {"XGBoost": [], "Logistic": []}
        oof_dev_losbin:  Dict[str, List[pd.DataFrame]] = {"XGBoost": [], "Logistic": []}
        hold_raw_losbin: Dict[str, List[pd.DataFrame]] = {"XGBoost": [], "Logistic": []}

        # MIごとに OOF 作成 → 後で単一校正器を学習
        for i, dt in enumerate(dt_list, 1):
            hold_mask = (dt[Config.TIME_FLAG_COL] == 1).to_numpy()
            idx_hold = dt.loc[hold_mask, Config.INDEX_COL].to_numpy()

            # ---- OUTCOME ----
            Xd_out, yd_out, Xh_out, _ = prepare_from_dt_timeaware(dt, label_col=Config.TARGET_COL)
            for mname in ["XGBoost", "Logistic"]:
                if mname == "XGBoost":
                    base_model, _ = build_estimator("XGBoost", stage)
                    base_model.set_params(**BEST_PARAMS_OUTCOME[mname])
                else:
                    base_model, _ = build_estimator("Logistic", stage)
                    base_model.set_params(**BEST_PARAMS_OUTCOME[mname])
                    if base_model.get_params().get("penalty", "l2") != "elasticnet":
                        base_model.set_params(l1_ratio=None)
                keep_cols = KEEP_OUTCOME
                # OOF
                oof_prob, pre_template = oof_predict_binary(base_model, keep_cols, Xd_out, yd_out,
                                                            n_splits=N_SPLITS, es_rounds=ES_ROUNDS, model_name=mname)
                oof_dev_outcome[mname].append(pd.DataFrame(oof_prob, index=Xd_out.index, columns=["p0", "p1"]))
                # HOLD raw
                proba_hold = fit_full_and_predict_hold(base_model, pre_template, Xd_out, yd_out, Xh_out,
                                                       es_rounds=ES_ROUNDS, task_tag="OUTCOME", model_tag=mname,
                                                       feat_names=list(Xd_out.columns), idx_hold=idx_hold, mi_id=i)
                hold_raw_outcome[mname].append(pd.DataFrame(proba_hold, index=Xh_out.index, columns=["p0", "p1"]))

            # ---- LOSBIN ----
            Xd_lb, yd_lb, Xh_lb, _ = prepare_from_dt_timeaware(dt, label_col=Config.LOS_BIN_COL)
            for mname in ["XGBoost", "Logistic"]:
                if mname == "XGBoost":
                    base_model, _ = build_estimator("XGBoost", stage)
                    base_model.set_params(**BEST_PARAMS_LOS[mname])
                else:
                    base_model, _ = build_estimator("Logistic", stage)
                    base_model.set_params(**BEST_PARAMS_LOS[mname])
                    if base_model.get_params().get("penalty", "l2") != "elasticnet":
                        base_model.set_params(l1_ratio=None)
                keep_cols = KEEP_LOSBIN
                # OOF
                oof_prob, pre_template = oof_predict_binary(base_model, keep_cols, Xd_lb, yd_lb,
                                                            n_splits=N_SPLITS, es_rounds=ES_ROUNDS, model_name=mname)
                oof_dev_losbin[mname].append(pd.DataFrame(oof_prob, index=Xd_lb.index, columns=["p0", "p1"]))
                # HOLD raw
                proba_hold = fit_full_and_predict_hold(base_model, pre_template, Xd_lb, yd_lb, Xh_lb,
                                                       es_rounds=ES_ROUNDS, task_tag="LOSBIN", model_tag=mname,
                                                       feat_names=list(Xd_lb.columns), idx_hold=idx_hold, mi_id=i)
                hold_raw_losbin[mname].append(pd.DataFrame(proba_hold, index=Xh_lb.index, columns=["p0", "p1"]))

        # ── MIプール & 単一校正器の学習（dev-OOF）
        dt0 = dt_list[0]
        hold_mask0 = (dt0[Config.TIME_FLAG_COL] == 1).to_numpy()
        dev_mask0  = ~hold_mask0
        index_hold = dt0.loc[hold_mask0, Config.INDEX_COL].to_numpy()

        y_true_outcome_dev = dt0.loc[dev_mask0, Config.TARGET_COL].astype(int).to_numpy()
        y_true_losbin_dev  = dt0.loc[dev_mask0, Config.LOS_BIN_COL].astype(int).to_numpy()

        y_true_outcome_hold = dt0.loc[hold_mask0, Config.TARGET_COL].astype(int).to_numpy()
        y_true_losbin_hold  = dt0.loc[hold_mask0, Config.LOS_BIN_COL].astype(int).to_numpy()
        y_true_los_hold     = dt0.loc[hold_mask0, Config.LOS_COL].astype(float).to_numpy()

        def pool_probs_on_index(df_list: List[pd.DataFrame], mask, task: str) -> np.ndarray:
            aligned = [df.reindex(dt0.index[mask]) for df in df_list]
            concat = pd.concat(aligned, axis=1)
            arr = concat.to_numpy().reshape(concat.shape[0], -1, 2)
            return arr.mean(axis=1)

        # calibrators を格納
        calibrators: Dict[Tuple[str, str], ProbCalibrator] = {}

        def train_and_apply_single_calibrator(task_name: str,
                                              oof_map: Dict[str, List[pd.DataFrame]],
                                              hold_map: Dict[str, List[pd.DataFrame]],
                                              y_dev_true: np.ndarray, y_hold_true: np.ndarray):
            class_rows: List[Dict[str, Any]] = []
            for mname in ["XGBoost", "Logistic"]:
                # dev OOF MI平均 → 単一校正器学習
                y_prob_dev_mipooled = pool_probs_on_index(oof_map[mname], dev_mask0, task_name)
                p1_dev = y_prob_dev_mipooled[:, 1]
                cal = ProbCalibrator(method=Config.CAL_METHOD)
                if Config.CAL_METHOD != "none":
                    cal.fit(y_dev_true, p1_dev)
                    save_calibrator_bundle(task_name, mname, Config.CAL_METHOD, cal)
                calibrators[(task_name, mname)] = cal

                # hold raw MI平均 → 単一校正器適用
                y_prob_hold_raw = pool_probs_on_index(hold_map[mname], hold_mask0, task_name)
                p1_raw = y_prob_hold_raw[:, 1]
                if Config.CAL_METHOD != "none":
                    p1_cal = cal.transform(p1_raw)
                    y_prob_hold_cal = np.vstack([1 - p1_cal, p1_cal]).T
                else:
                    y_prob_hold_cal = y_prob_hold_raw

                # 評価・保存
                def summarize(y_true, y_prob, tag):
                    cm_final = confusion_matrix(y_true, np.argmax(y_prob, axis=1))
                    pd.DataFrame(
                        cm_final,
                        index=Config.CLASS_NAMES if task_name == "OUTCOME" else ["≤21日", "＞21日"],
                        columns=Config.CLASS_NAMES if task_name == "OUTCOME" else ["≤21日", "＞21日"],
                    ).to_csv(Config.RESULT_DIR / f"{task_name}_{mname}_{tag}_cm.csv", encoding="utf-8-sig")
                    save_confusion_matrix_plot(cm_final,
                                               Config.CLASS_NAMES if task_name == "OUTCOME" else ["≤21日", "＞21日"],
                                               mname + f"({tag})", task_name)
                    ci = get_bootstrap_ci(y_true, y_prob, n_boot=N_BOOT)
                    slope, citl = calc_calibration_slope_citl(y_true, y_prob[:, 1])
                    row = {
                        "Task": task_name, "Model": f"{mname} ({tag})", "N_hold": int(len(y_true)),
                        "AUROC": ci["AUROC"][0], "AUROC_low": ci["AUROC"][1], "AUROC_high": ci["AUROC"][2],
                        "AUPRC": ci["AP"][0],   "AUPRC_low": ci["AP"][1],   "AUPRC_high": ci["AP"][2],
                        "Brier": ci["Brier"][0],"Brier_low": ci["Brier"][1],"Brier_high": ci["Brier"][2],
                        "CalibSlope": slope, "CalibSlope_low": ci["Slope"][1], "CalibSlope_high": ci["Slope"][2],
                        "CITL": citl, "CITL_low": ci["CITL"][1], "CITL_high": ci["CITL"][2]
                    }
                    plot_calibration_curve_binary(
                        y_true, y_prob[:, 1],
                        Config.RESULT_DIR / f"calibration_curve_{task_name}_{mname}_{tag}_PLOTTED.png",
                        f"Calibration ({task_name}, {mname}, {tag})", label=mname
                    )
                    return row

                # raw_MIPOOL
                class_rows.append(summarize(y_hold_true, y_prob_hold_raw, "raw_mipooled"))
                save_predictions_csv(task_name, mname, "raw_mipooled", index_hold, y_hold_true, y_prob_hold_raw[:, 1])

                # calibrated after pool
                tag_cal = f"postpool_{Config.CAL_METHOD}_single"
                class_rows.append(summarize(y_hold_true, y_prob_hold_cal, tag_cal))
                save_predictions_csv(task_name, mname, tag_cal, index_hold, y_hold_true, y_prob_hold_cal[:, 1])

            return class_rows

        # OUTCOME/LOSBIN の校正・評価
        class_rows_out = train_and_apply_single_calibrator("OUTCOME", oof_dev_outcome, hold_raw_outcome,
                                                           y_true_outcome_dev, y_true_outcome_hold)
        class_rows_lb  = train_and_apply_single_calibrator("LOSBIN",  oof_dev_losbin,  hold_raw_losbin,
                                                           y_true_losbin_dev,  y_true_losbin_hold)

        # ── LOS回帰（gateは校正後LOSBIN-XGBの確率）
        hold_pred_los_reg_xgb_list: List[pd.DataFrame] = []
        hold_pred_los_reg2s_xgb_list: List[pd.DataFrame] = []

        for i, dt in enumerate(dt_list, 1):
            _, hold_idx_arr = split_indices_from_first(dt_list)
            idx_hold_i = dt.loc[hold_idx_arr, Config.INDEX_COL].to_numpy()

            Xd_r, _, Xh_r, _ = prepare_from_dt_timeaware(dt, label_col=Config.LOS_COL)
            ylos_dev = dt.loc[Xd_r.index, Config.LOS_COL].astype(float).to_numpy()

            # 回帰（単発）: 前処理→fit→HOLD 予測
            pre_r = make_runtime_preprocess_from_keep(KEEP_LOSREG).fit(Xd_r, ylos_dev * 0)
            model_reg = clone(pipe_reg)
            ytr_log = np.log1p(ylos_dev)
            model_reg.fit(Xd_r, ytr_log)
            yhat_los_xgb = np.expm1(model_reg.predict(Xh_r)).clip(0, None)
            hold_pred_los_reg_xgb_list.append(pd.DataFrame({f"LOS_pred_MI{i}": yhat_los_xgb}, index=Xh_r.index))
            # 保存（前処理出力も保存）
            Xh_t_single = pre_r.transform(Xh_r)
            save_model_bundle("LOSREG", "XGBRegressor", "raw", model_reg, pre_r, Xh_t_single, idx_hold_i, Xd_r.columns, mi_id=i)

            # ── 二段階: gate=LOSBIN XGB（postpool単一校正）を使用
            bin_dev = dt.loc[Xd_r.index, Config.LOS_BIN_COL].astype(int).to_numpy()
            idx_short = np.where(bin_dev == 0)[0]
            idx_long  = np.where(bin_dev == 1)[0]

            # 前処理済みを再利用（DataFrame）
            Xr_t = pre_r.transform(Xd_r)  # DataFrame
            Xh_t = pre_r.transform(Xh_r)  # DataFrame

            # 短期モデル
            reg_short, _ = build_regressor("FINAL")
            params_from_pipe = {k.replace("model__",""): v
                                for k, v in model_reg.get_params().items() if k.startswith("model__")}
            reg_short.set_params(**params_from_pipe)
            reg_short_fit, _ = fit_with_gpu_fallback_xgb(
                clone(reg_short),
                Xr_t.iloc[idx_short].values,
                np.log1p(ylos_dev[idx_short])
            )

            # 長期モデル
            reg_long, _ = build_regressor("FINAL")
            reg_long.set_params(**params_from_pipe)
            reg_long_fit, _ = fit_with_gpu_fallback_xgb(
                clone(reg_long),
                Xr_t.iloc[idx_long].values,
                np.log1p(ylos_dev[idx_long])
            )

            # 予測（前処理済みを使う）
            yhat_short = np.expm1(reg_short_fit.predict(Xh_t.values)).clip(0, None)
            yhat_long  = np.expm1(reg_long_fit.predict(Xh_t.values)).clip(0, None)

            # gate確率（LOSBIN, XGBoost, postpool calibrated）
            p_long_df_list = hold_raw_losbin["XGBoost"]  # raw per-MI
            y_prob_lb_hold_raw_miavg = pool_probs_on_index(p_long_df_list, hold_mask0, "LOSBIN")
            p1_lb_raw = y_prob_lb_hold_raw_miavg[:, 1]
            cal_lb = calibrators[("LOSBIN", "XGBoost")]
            if Config.CAL_METHOD != "none":
                p_long = cal_lb.transform(p1_lb_raw)
            else:
                p_long = p1_lb_raw

            # 二段階結合
            yhat_2s = (1 - p_long) * yhat_short + p_long * yhat_long
            hold_pred_los_reg2s_xgb_list.append(pd.DataFrame({f"LOS2S_pred_MI{i}": yhat_2s}, index=Xh_r.index))

            # 保存（短期・長期モデル）
            save_model_bundle("LOS2S_SHORT", "XGBRegressor", "raw", reg_short_fit, pre_r, Xh_t, idx_hold_i, Xd_r.columns, mi_id=i)
            save_model_bundle("LOS2S_LONG",  "XGBRegressor", "raw", reg_long_fit,  pre_r, Xh_t, idx_hold_i, Xd_r.columns, mi_id=i)

        # ── 集計（分類 & 回帰）
        class_rows = class_rows_out + class_rows_lb
        reg_rows: List[Dict[str, Any]] = []

        # 回帰: 単発 MI平均
        los_concat = pd.concat(hold_pred_los_reg_xgb_list, axis=1)
        los_mean   = los_concat.mean(axis=1).to_frame("LOS_pred")
        yhat_los_hold_xgb = los_mean.loc[los_mean.index, "LOS_pred"].to_numpy()
        save_los_predictions_named("LOSREG", "XGBRegressor", index_hold, y_true_los_hold, yhat_los_hold_xgb)

        # 二段階 MI平均
        los2s_concat = pd.concat(hold_pred_los_reg2s_xgb_list, axis=1)
        los2s_mean   = los2s_concat.mean(axis=1).to_frame("LOS_pred")
        yhat_los_hold_2s = los2s_mean.loc[los2s_mean.index, "LOS_pred"].to_numpy()
        save_los_predictions_named("LOS2S", "XGBRegressor", index_hold, y_true_los_hold, yhat_los_hold_2s)

        # 回帰メトリクス
        def _mape(y_true, y_pred, eps=1e-6):
            y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
            denom = np.maximum(np.abs(y_true), eps)
            return float(np.mean(np.abs((y_pred - y_true) / denom)))

        rmse_x  = rmse_score(y_true_los_hold, yhat_los_hold_xgb)
        mae_x   = float(np.mean(np.abs(y_true_los_hold - yhat_los_hold_xgb)))
        medae_x = median_absolute_error(y_true_los_hold, yhat_los_hold_xgb)
        mape_x  = _mape(y_true_los_hold, yhat_los_hold_xgb)
        r2_x    = r2_score(y_true_los_hold, yhat_los_hold_xgb)
        reg_rows.append({"Task": "LOSREG", "Model": "XGBRegressor (single)", "N_hold": int(len(y_true_los_hold)),
                         "RMSE": rmse_x, "MAE": mae_x, "MedAE": medae_x, "MAPE": mape_x, "R2": r2_x})

        rmse_2s  = rmse_score(y_true_los_hold, yhat_los_hold_2s)
        mae_2s   = float(np.mean(np.abs(y_true_los_hold - yhat_los_hold_2s)))
        medae_2s = median_absolute_error(y_true_los_hold, yhat_los_hold_2s)
        mape_2s  = _mape(y_true_los_hold, yhat_los_hold_2s)
        r2_2s    = r2_score(y_true_los_hold, yhat_los_hold_2s)
        reg_rows.append({"Task": "LOS2S", "Model": "XGBRegressor (two-stage, gate=LOSBIN-XGB postpool-calibrated)", "N_hold": int(len(y_true_los_hold)),
                         "RMSE": rmse_2s, "MAE": mae_2s, "MedAE": medae_2s, "MAPE": mape_2s, "R2": r2_2s})

        # 出力集約
        pd.DataFrame({"device": device_records}).to_csv(Config.RESULT_DIR / "device_meta.csv", index=False, encoding="utf-8-sig")
        class_df = pd.DataFrame(class_rows); class_df.to_csv(Config.RESULT_DIR / "Classification.csv", index=False, encoding="utf-8-sig")
        reg_df   = pd.DataFrame(reg_rows);   reg_df.to_csv(Config.RESULT_DIR   / "Regression.csv",    index=False, encoding="utf-8-sig")
        with pd.ExcelWriter(Config.RESULT_DIR / "all_metrics_summary.xlsx", engine="xlsxwriter") as writer:
            class_df.to_excel(writer, sheet_name="Classification", index=False)
            reg_df.to_excel(writer, sheet_name="Regression", index=False)
            pd.DataFrame({"device": device_records}).to_excel(writer, sheet_name="Meta", index=False)

        print("完了。出力先:", Config.RESULT_DIR.resolve())
        print("Predictions:", (Config.RESULT_DIR / Config.PRED_DIRNAME).resolve())
        print("Artifacts:", (Config.RESULT_DIR / Config.ARTIFACT_DIRNAME).resolve())
        print("集約Excel:", (Config.RESULT_DIR / "all_metrics_summary.xlsx").resolve())

    return {
        "BEST_PARAMS_OUTCOME": locals().get("BEST_PARAMS_OUTCOME", {}),
        "BEST_PARAMS_LOS":     locals().get("BEST_PARAMS_LOS", {}),
        "BEST_PARAMS_REG":     locals().get("BEST_PARAMS_REG_XGB", {})
    }

# ─────────────────────────────────────────────
# メイン
# ─────────────────────────────────────────────
if __name__ == "__main__":
    logger.info("Training config (Two-stage tuning: TUNE→FINAL/QUICK; OOF→Single Calibrator→HOLD)")
    raw = pd.read_excel(RAW_PATH)
    logger.info(f"Raw shape: {raw.shape}")

    if RUN_STAGE in ("TUNE", "ALL"):
        _ = run_stage("TUNE", raw)

    if RUN_STAGE in ("FINAL", "ALL", "QUICK"):
        _ = run_stage("FINAL" if RUN_STAGE=="FINAL" else ("QUICK" if RUN_STAGE=="QUICK" else "FINAL"), raw)
        # "ALL" の場合は FINAL を実行。QUICK の場合は QUICK を実行。


2025-09-12 08:12:22,781 [INFO] [FORCE] CSV未指定: グローバル強制保持のみ
2025-09-12 08:12:22,802 [INFO] Training config (Two-stage tuning: TUNE→FINAL/QUICK; OOF→Single Calibrator→HOLD)
2025-09-12 08:12:23,914 [INFO] Raw shape: (3000, 76)
2025-09-12 08:12:23,914 [INFO] === RUN STAGE: FINAL ===
2025-09-12 08:13:07,548 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,550 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,552 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,555 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,558 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,560 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,561 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,565 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,566 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,569 [INFO] Temporal split[転帰]: dev=2205, hold=795
2025-09-12 08:13:07,

c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-09-12 08:13:11,631] A new study created in memory with name: no-name-a084f324-92f0-4819-ab39-3328df362e42
c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\xgboost\training.py:183: UserWarning: [08:13:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\xgboost\training.py:183: UserWarning: [08:13:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are 

2025-09-12 08:25:15,928 [INFO] [FINAL-TUNED - 転帰 - XGBoost] {'max_depth': 8, 'learning_rate': 0.03555507439791321, 'min_child_weight': 4, 'gamma': 2.094827147518615, 'subsample': 0.861417591408874, 'colsample_bytree': 0.7057422017430685, 'reg_alpha': 0.0022302298109150983, 'reg_lambda': 0.6751259620658274}


c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\optuna\_experimental.py:31: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-09-12 08:25:15,931] A new study created in memory with name: no-name-0b715e81-cc99-436c-8029-b4c2f482e756
[I 2025-09-12 08:26:21,920] Trial 0 finished with value: 0.8083909058842392 and parameters: {'model__C': 0.0074593432857265485, 'model__penalty': 'elasticnet', 'model__l1_ratio': 0.2}. Best is trial 0 with value: 0.8083909058842392.
[I 2025-09-12 08:26:25,733] Trial 1 finished with value: 0.7899026216816692 and parameters: {'model__C': 0.00019517224641449495, 'model__penalty': 'elasticnet', 'model__l1_ratio': 0.8}. Best is trial 0 with value: 0.8083909058842392

2025-09-12 08:38:38,152 [INFO] [FINAL-TUNED - 転帰 - Logistic] {'C': 0.0074593432857265485, 'penalty': 'elasticnet', 'l1_ratio': 0.2}
2025-09-12 08:38:38,154 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,155 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,157 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,158 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,160 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,161 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,163 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,166 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,168 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,170 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,171 [INFO] Temporal split[LOS_bin]: dev=2205, hold=795
2025-09-12 08:38:38,173 [INFO] Temporal spl

...

2025-09-12 09:48:07,852 [INFO] [ARTIFACT] Saved → LOS2S_SHORT__XGBRegressor__raw__MI14.pkl (feat=63, holdN=795)
2025-09-12 09:48:07,899 [INFO] [ARTIFACT] Saved → LOS2S_LONG__XGBRegressor__raw__MI14.pkl (feat=63, holdN=795)
2025-09-12 09:48:07,903 [INFO] Temporal split[在院日数]: dev=2205, hold=795


c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:48:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2025-09-12 09:48:09,050 [INFO] [ARTIFACT] Saved → LOSREG__XGBRegressor__raw__MI15.pkl (feat=63, holdN=795)


c:\Users\tears\anaconda3\envs\py311\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:48:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2025-09-12 09:48:10,390 [INFO] [ARTIFACT] Saved → LOS2S_SHORT__XGBRegressor__raw__MI15.pkl (feat=63, holdN=795)
2025-09-12 09:48:10,435 [INFO] [ARTIFACT] Saved → LOS2S_LONG__XGBRegressor__raw__MI15.pkl (feat=63, holdN=795)
2025-09-12 09:48:10,444 [INFO] [PRED] Saved → LOSREG__XGBRegressor__raw__hold.csv
2025-09-12 09:48:10,451 [INFO] [PRED] Saved → LOS2S__XGBRegressor__raw__hold.csv
完了。出力先: C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\results_20250912_0812
Predictions: C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\results_20250912_0812\predictions
Artifacts: C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\results_20250912_0812\artifacts
集約Excel: C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\results_20250912_0812\all_metrics_summary.xlsx


In [6]:
# -*- coding: utf-8 -*-
# ============================================================
# figure_block_optimal_v3.py
# 目的:
#  - OUTCOME/LOSBIN：XGBoost と Logistic を同一図に重ねたカラー図を
#    ROC / PR / Calibration / DCA で「1枚ずつ」出力
#  - SHAP：artifacts の feature_names を優先して取得し、MI平均で描画
# 入力: results_*/predictions/*.csv, results_*/artifacts/*.pkl
# 出力: results_*/figures/
# 実行例:
#   python figure_block_optimal_v3.py --result_dir results_20250909_1030 \
#     --tasks OUTCOME,LOSBIN --variant postpool_sigmoid_single \
#     --dpi 600 --cal_bins 10 --n_boot 1000 --seed 42 --shap_topn 25
# 備考: Jupyter/VSCode でも parse_known_args で未知引数を無視
# ============================================================

from __future__ import annotations
import argparse, re, sys
from pathlib import Path
from typing import List, Tuple, Dict, Any
import japanize_matplotlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

import shap
import joblib

# ----------------------------
# 既定設定
# ----------------------------
DEF_TASKS   = ["OUTCOME", "LOSBIN"]
DEF_MODELS  = ["XGBoost", "Logistic"]
DEF_VARIANT = "postpool_sigmoid_single"

CLASS_NAMES = {
    "OUTCOME": ["自宅", "その他"],
    "LOSBIN" : ["≤21日", "＞21日"]
}

# カラー（色 + 線種）。学会・論文で見やすい定番配色。
MODEL_STYLES = {
    "XGBoost":  dict(color="#1f77b4", linewidth=2.2, linestyle="-"),   # blue
    "Logistic": dict(color="#ff7f0e", linewidth=2.0, linestyle="--"),  # orange
}
CAL_MARKERS = {"XGBoost": "o", "Logistic": "s"}  # calibration でのみ使用

# ----------------------------
# 汎用
# ----------------------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def read_pred_csv(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    y_true = df["y_true"].to_numpy().astype(int)
    p1 = df["y_prob"].to_numpy().astype(float)
    return y_true, p1

def list_pred_paths(pred_dir: Path, task: str, variant: str, models: List[str]) -> Dict[str, Path]:
    out = {}
    for m in models:
        fn = pred_dir / f"{task}__{m}__{variant}__hold.csv"
        if fn.exists():
            out[m] = fn
    return out

def ci_from_boot(xs: np.ndarray, alpha=0.05):
    xs = xs.astype(float)
    return float(np.nanmean(xs)), float(np.nanpercentile(xs, 100*alpha/2)), float(np.nanpercentile(xs, 100*(1-alpha/2)))

def calibration_slope_citl(y_true: np.ndarray, p1: np.ndarray) -> Tuple[float, float]:
    p = np.clip(p1, 1e-6, 1-1e-6)
    logit = np.log(p/(1-p)).reshape(-1,1)
    y = y_true.astype(int).ravel()
    lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)
    lr.fit(logit, y)
    slope = float(lr.coef_.ravel()[0])
    citl  = float(lr.intercept_.ravel()[0])
    return slope, citl

def bootstrap_metric(y: np.ndarray, p1: np.ndarray, n_boot=1000, seed=42) -> Dict[str, Tuple[float,float,float]]:
    rng = np.random.default_rng(seed)
    n = len(y)
    aurocs, aprs, briers, slopes, citls = [], [], [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yy = y[idx]; pp = p1[idx]
        try: aurocs.append(roc_auc_score(yy, pp))
        except: aurocs.append(np.nan)
        try: aprs.append(average_precision_score(yy, pp))
        except: aprs.append(np.nan)
        try: briers.append(float(np.mean((pp - yy)**2)))
        except: briers.append(np.nan)
        try:
            s, c = calibration_slope_citl(yy, pp)
            slopes.append(s); citls.append(c)
        except:
            slopes.append(np.nan); citls.append(np.nan)
    return {
        "AUROC": ci_from_boot(np.array(aurocs)),
        "AUPRC": ci_from_boot(np.array(aprs)),
        "Brier": ci_from_boot(np.array(briers)),
        "Slope": ci_from_boot(np.array(slopes)),
        "CITL" : ci_from_boot(np.array(citls)),
    }

# ----------------------------
# 図：各種 1枚ずつ出力（XGB vs LR を重ね）
# ----------------------------
def fig_roc(task: str, curves: Dict[str, Dict[str, Any]], out_png: Path, dpi=600):
    plt.figure(figsize=(6.2, 5.6))
    plt.plot([0,1],[0,1], linestyle=":", color="#888888", linewidth=1)
    for m, d in curves.items():
        y, p = d["y"], d["p"]
        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        st = MODEL_STYLES.get(m, {})
        plt.plot(fpr, tpr, label=f"{m} (AUC={auc:.3f})", **st)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC – {task}")
    plt.legend(loc="lower right", frameon=False)
    plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

def fig_pr(task: str, curves: Dict[str, Dict[str, Any]], out_png: Path, dpi=600):
    plt.figure(figsize=(6.2, 5.6))
    for m, d in curves.items():
        y, p = d["y"], d["p"]
        prec, rec, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        st = MODEL_STYLES.get(m, {})
        plt.plot(rec, prec, label=f"{m} (AP={ap:.3f})", **st)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"Precision–Recall – {task}")
    plt.legend(loc="lower left", frameon=False)
    plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

def fig_calibration(task: str, curves: Dict[str, Dict[str, Any]], out_png: Path, cal_bins=10, dpi=600):
    plt.figure(figsize=(6.2, 5.6))
    plt.plot([0,1],[0,1], linestyle=":", color="#888888", linewidth=1, label="Perfect")
    for m, d in curves.items():
        y, p = d["y"], d["p"]
        ob, pr = calibration_curve(y, p, n_bins=cal_bins, strategy="quantile")
        slope, citl = calibration_slope_citl(y, p)
        st = MODEL_STYLES.get(m, {}).copy()
        st.pop("marker", None)  # marker 二重指定を回避
        mk = CAL_MARKERS.get(m, "o")
        plt.plot(pr, ob, marker=mk, markersize=4.0, **st,
                 label=f"{m} (Slope={slope:.2f}, CITL={citl:.2f})")
    plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency")
    plt.title(f"Calibration – {task}")
    plt.legend(loc="upper left", frameon=False)
    plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

def decision_curve(y_true: np.ndarray, p1: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    y = y_true.astype(int).ravel(); N = len(y)
    out = []
    for pt in thresholds:
        yhat = (p1 >= pt).astype(int)
        TP = np.sum((yhat == 1) & (y == 1))
        FP = np.sum((yhat == 1) & (y == 0))
        nb_model = (TP/N) - (FP/N) * (pt/(1-pt))
        out.append(nb_model)
    return np.array(out, float)

def fig_dca(task: str, curves: Dict[str, Dict[str, Any]], out_png: Path, dpi=600):
    thresholds = np.linspace(0.01, 0.99, 99)
    plt.figure(figsize=(6.6, 5.6))
    # treat-none / treat-all
    y0 = next(iter(curves.values()))["y"]
    prev = y0.mean()
    nb_all = prev - (1 - prev) * (thresholds/(1-thresholds))
    plt.plot(thresholds, np.zeros_like(thresholds), linestyle=":", color="#888888", label="Treat-none")
    plt.plot(thresholds, nb_all, linestyle="--", color="#888888", label="Treat-all")
    # models
    for m, d in curves.items():
        y, p = d["y"], d["p"]
        st = MODEL_STYLES.get(m, {})
        nb = decision_curve(y, p, thresholds)
        plt.plot(thresholds, nb, label=m, **st)
    plt.xlabel("Threshold probability"); plt.ylabel("Net benefit")
    plt.title(f"Decision Curve Analysis – {task}")
    plt.legend(loc="upper right", frameon=False)
    plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

# ----------------------------
# SHAP（artifacts→MI平均、feature_namesをpklから優先取得）
# ----------------------------
def find_artifacts(art_dir: Path, task: str, model: str, calib_tag="raw_full_dev") -> List[Path]:
    patt = re.compile(rf"^{re.escape(task)}__{re.escape(model)}__{re.escape(calib_tag)}(__MI\d+)?\.pkl$")
    return sorted([p for p in art_dir.glob("*.pkl") if patt.match(p.name)], key=lambda x: x.name)

def safe_feature_names(bundle: Dict[str,Any], pre) -> List[str]:
    if "feature_names" in bundle and bundle["feature_names"] is not None:
        return [str(x) for x in bundle["feature_names"]]
    for obj in [pre, getattr(pre, "named_steps", {}).get("keep", None)]:
        if obj is None: continue
        if hasattr(obj, "get_feature_names_out"):
            try:
                names = obj.get_feature_names_out()
            except TypeError:
                names = obj.get_feature_names_out(None)
            return [str(x) for x in names]
        if hasattr(obj, "get_feature_names"):
            return [str(x) for x in obj.get_feature_names()]
        if hasattr(obj, "columns"):
            return [str(x) for x in getattr(obj, "columns")]
    raise RuntimeError("feature_names の取得に失敗しました。")

def shap_from_artifacts(files: List[Path]) -> Tuple[np.ndarray, np.ndarray, List[str], np.ndarray, np.ndarray]:
    if len(files) == 0:
        raise FileNotFoundError("SHAP artifacts not found.")
    shap_list, base_list, feat_list, x_list, idx_list = [], [], [], [], []
    for fp in files:
        b: Dict[str,Any] = joblib.load(fp)
        model = b["model"]; pre = b["pre"]
        Xh = np.array(b["X_hold_t"], dtype=float)
        idx = np.array(b["index_hold"])
        feat = safe_feature_names(b, pre)

        model_name = type(model).__name__
        if "XGB" in model_name:
            expl = shap.TreeExplainer(model, feature_names=feat)
            sv = expl.shap_values(Xh); ev = expl.expected_value
            if isinstance(sv, list): sv = sv[-1]
            if isinstance(ev, list): ev = ev[-1]
        elif "LogisticRegression" in model_name:
            rng = np.random.default_rng(42)
            bg = Xh[rng.choice(len(Xh), size=min(200, len(Xh)), replace=False)]
            try:
                expl = shap.LinearExplainer(model, bg, feature_names=feat)
                sv = expl.shap_values(Xh); ev = expl.expected_value
                if isinstance(sv, list): sv = sv[-1]
                if isinstance(ev, list): ev = ev[-1]
            except Exception:
                expl = shap.KernelExplainer(model.predict_proba, bg)
                sv = expl.shap_values(Xh, nsamples="auto"); ev = expl.expected_value
                if isinstance(sv, list): sv = sv[-1]
                if isinstance(ev, list): ev = ev[-1]
        else:
            raise NotImplementedError(f"Unsupported model for SHAP: {model_name}")

        feat_list.append(feat)
        shap_list.append(np.asarray(sv, float))
        base_list.append(np.full(Xh.shape[0], float(ev)))
        x_list.append(Xh); idx_list.append(idx.astype(int))

    feat_ref = feat_list[0]
    for f in feat_list[1:]:
        if list(f) != list(feat_ref):
            raise RuntimeError("MI間で feature_names が一致しません。")
    idx_ref = idx_list[0]
    pos_map = {v: i for i, v in enumerate(idx_ref)}
    aligned_sv = [shap_list[0]]; aligned_ev = [base_list[0]]
    for i in range(1, len(files)):
        idx_i = idx_list[i]
        reorder = [pos_map[v] for v in idx_i]
        inv = np.empty_like(reorder); inv[np.array(reorder)] = np.arange(len(reorder))
        aligned_sv.append(shap_list[i][inv]); aligned_ev.append(base_list[i][inv])

    shap_mean = np.mean(np.stack(aligned_sv, axis=0), axis=0)
    base_mean = np.mean(np.stack(aligned_ev, axis=0), axis=0)
    return shap_mean, base_mean, feat_ref, x_list[0], idx_ref

def save_global_importance(shap_values: np.ndarray, feature_names: List[str], out_csv: Path, topn: int):
    imp = np.abs(shap_values).mean(axis=0)
    df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": imp})
    df = df.sort_values("mean_abs_shap", ascending=False, ignore_index=True)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return df.head(topn)

def plot_shap_beeswarm(shap_values: np.ndarray, X_input: np.ndarray, feature_names: List[str], out_png: Path, max_display: int, dpi=300, sample_cap=20000, seed=42):
    rng = np.random.default_rng(seed)
    Xp, Sp = X_input, shap_values
    if X_input.shape[0] > sample_cap:
        idx = rng.choice(X_input.shape[0], size=sample_cap, replace=False)
        Xp = X_input[idx]; Sp = shap_values[idx]
    plt.figure()
    shap.summary_plot(Sp, Xp, feature_names=feature_names, show=False, max_display=max_display)
    plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

def plot_shap_bar(shap_values: np.ndarray, feature_names: List[str], out_png: Path, max_display: int, dpi=300):
    plt.figure()
    shap.summary_plot(shap_values, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
    plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

def plot_shap_force(expected_value: float, shap_row: np.ndarray, feature_names: List[str], out_png: Path, dpi=300):
    try:
        plt.figure(figsize=(10,1.8))
        _ = shap.plots._force.AdditiveForceVisualizer(expected_value, shap_row, feature_names=feature_names, matplotlib=True)
        plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()
    except Exception:
        shap.force_plot(expected_value, shap_row, feature_names=feature_names, matplotlib=True, show=False)
        fig = plt.gcf(); fig.set_size_inches(10,1.8)
        plt.tight_layout(); plt.savefig(out_png, dpi=dpi, bbox_inches="tight"); plt.close()

# ----------------------------
# CLI
# ----------------------------
def cli(argv: list[str] | None = None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--result_dir", type=str, required=False, default=None,
                    help="results_*。未指定なら最新を自動検出")
    ap.add_argument("--tasks", type=str, default=",".join(DEF_TASKS))
    ap.add_argument("--variant", type=str, default=DEF_VARIANT,
                    help="predictions のバリアント名（例：postpool_sigmoid_single）")
    ap.add_argument("--dpi", type=int, default=600)
    ap.add_argument("--cal_bins", type=int, default=10)
    ap.add_argument("--n_boot", type=int, default=1000)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--shap_topn", type=int, default=25)
    args, unknown = ap.parse_known_args(argv)
    if unknown:
        print(f"[INFO] Ignored unknown args: {unknown}")

    # results_* 解決
    if args.result_dir is None:
        cands = sorted([p for p in Path(".").glob("results_*") if p.is_dir()], key=lambda x: x.name)
        if not cands:
            print("[ERR] results_* が見つかりません。--result_dir を指定してください。")
            sys.exit(1)
        result_dir = cands[-1]
    else:
        result_dir = Path(args.result_dir)

    pred_dir = result_dir / "predictions"
    art_dir  = result_dir / "artifacts"
    fig_dir  = result_dir / "figures"
    ensure_dir(fig_dir)

    tasks = [t.strip() for t in args.tasks.split(",") if t.strip()]
    models = DEF_MODELS

    # ===== 図（各タイプを1枚ずつ）＋ 要約表 =====
    summary_rows: List[Dict[str, Any]] = []
    for task in tasks:
        paths = list_pred_paths(pred_dir, task, args.variant, models)
        if len(paths) < 2:
            print(f"[WARN] {task} の {args.variant} で両モデルの予測が揃っていません。skipped.")
            continue

        curves: Dict[str, Dict[str, Any]] = {}
        for m in models:
            y, p = read_pred_csv(paths[m])
            curves[m] = {"y": y, "p": p, "label": m}
            ci = bootstrap_metric(y, p, n_boot=args.n_boot, seed=args.seed)
            summary_rows.append({
                "Task": task, "Variant": args.variant, "Model": m, "N": len(y),
                "AUROC": ci["AUROC"][0], "AUROC_low": ci["AUROC"][1], "AUROC_high": ci["AUROC"][2],
                "AUPRC": ci["AUPRC"][0], "AUPRC_low": ci["AUPRC"][1], "AUPRC_high": ci["AUPRC"][2],
                "Brier": ci["Brier"][0], "Brier_low": ci["Brier"][1], "Brier_high": ci["Brier"][2],
                "CalibSlope": ci["Slope"][0], "CalibSlope_low": ci["Slope"][1], "CalibSlope_high": ci["Slope"][2],
                "CITL": ci["CITL"][0], "CITL_low": ci["CITL"][1], "CITL_high": ci["CITL"][2],
            })

        # 個別図を出力
        fig_roc(task, curves, fig_dir / f"ROC_{task}_{args.variant}__XGB_vs_LR.png", dpi=args.dpi)
        fig_pr(task, curves,  fig_dir / f"PR_{task}_{args.variant}__XGB_vs_LR.png",  dpi=args.dpi)
        fig_calibration(task, curves, fig_dir / f"CAL_{task}_{args.variant}__XGB_vs_LR.png",
                        cal_bins=args.cal_bins, dpi=args.dpi)
        fig_dca(task, curves, fig_dir / f"DCA_{task}_{args.variant}__XGB_vs_LR.png", dpi=args.dpi)

    if summary_rows:
        pd.DataFrame(summary_rows).to_csv(fig_dir / f"metrics_summary_{args.variant}_XGB_vs_LR.csv",
                                          index=False, encoding="utf-8-sig")

    # ===== SHAP（各タスク×各モデルで個別PNG/CSV）=====
    for task in tasks:
        for model in models:
            arts = find_artifacts(art_dir, task, model, calib_tag="raw_full_dev")
            if not arts:
                continue
            try:
                sv, base, feat, Xh, idx = shap_from_artifacts(arts)
            except Exception as e:
                print(f"[WARN] SHAP skipped: {task}/{model} reason={e}")
                continue

            gi_csv = fig_dir / f"SHAP_{task}_{model}_global_importance.csv"
            _ = save_global_importance(sv, feat, gi_csv, topn=args.shap_topn)
            plot_shap_beeswarm(sv, Xh, feat, fig_dir / f"SHAP_{task}_{model}_beeswarm.png",
                               max_display=args.shap_topn, dpi=300)
            plot_shap_bar(sv, feat, fig_dir / f"SHAP_{task}_{model}_bar.png",
                          max_display=args.shap_topn, dpi=300)
            plot_shap_force(float(base[0]), sv[0], feat,
                            fig_dir / f"SHAP_{task}_{model}_force_INDEX{idx[0]}.png", dpi=300)

    print("=== 完了 ===")
    print("Figures:", fig_dir.resolve())

if __name__ == "__main__":
    cli()


[INFO] Ignored unknown args: ['--f=c:\\Users\\tears\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3334ee321c1796ee91ec405799505dd2b433a520a.json']


C:\Users\tears\AppData\Local\Temp\ipykernel_14396\1475661849.py:288: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(Sp, Xp, feature_names=feature_names, show=False, max_display=max_display)
C:\Users\tears\AppData\Local\Temp\ipykernel_14396\1475661849.py:293: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
C:\Users\tears\AppData\Local\Temp\ipykernel_14396\1475661849.py:288: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the gl

=== 完了 ===
Figures: C:\Users\tears\Desktop\Study\2025\12_CI\006_ML3\results_20250912_0812\figures
